# Bengaluru Climate Risk Scanner
## Notebook 01: Data download and setup (Phase 1)

This notebook downloads and prepares every base layer the project needs:

| Output | Source | Saved as |
|---|---|---|
| City boundary | OpenStreetMap | `bengaluru.gpkg` → layer `bbmp_boundary` |
| Analysis area (city + 5 km buffer) | Derived | `bengaluru.gpkg` → layer `aoi` |
| Ward boundaries (optional) | DataMeet on GitHub | `bengaluru.gpkg` → layer `wards` |
| Lakes, tanks and wetlands | OpenStreetMap | `bengaluru.gpkg` → layer `lakes` |
| Drains, streams, canals | OpenStreetMap | `bengaluru.gpkg` → layer `drains` |
| Ground elevation, 30 m | FABDEM (Copernicus DEM with buildings and trees removed) via Earth Engine | `dem_30m.tif` |
| Built-up share, 30 m | ESA WorldCover via Earth Engine | `builtup_30m.tif` |
| Sentinel-1 scene list, Aug–Sep 2022 | Earth Engine | `s1_scenes_2022.csv` |

Everything is saved to Google Drive in `MyDrive/bengaluru-climate-risk/`, together with a `config.json` that every later notebook reads.

**Why a buffer?** Lakes, drains and terrain just outside the city still affect properties near the edge (Varthur and Sarjapur Road are on the southeast edge). So vectors and rasters cover the city plus a margin. Each vector feature gets an `in_city` flag.

**Before you start:** you need a Google Earth Engine Cloud project. Register one for free (noncommercial use) at https://code.earthengine.google.com/register, then paste its project ID into the config cell below.

Run the cells from top to bottom.

## 1. Install libraries

In [ ]:
!pip install -q osmnx geemap rasterio

## 2. Connect Google Drive and create the project folders
Colab deletes its own files when the session ends, so everything important is saved to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
RAW = f'{PROJECT_DIR}/data/raw'          # downloaded inputs
OUT = f'{PROJECT_DIR}/data/processed'    # final layers for GeoServer
WORK = '/content/work'                   # fast local scratch space

for d in (RAW, OUT, WORK):
    os.makedirs(d, exist_ok=True)

print('Project folder:', PROJECT_DIR)

## 3. Project configuration
This is the only cell you edit. Every later notebook reads `config.json`, so changing the city here (query, bounding box, dates) is all it takes to rerun the whole pipeline on another city.

In [ ]:
import json

CONFIG = {
    "city": "Bengaluru",
    "osm_query": "Bengaluru, Karnataka, India",
    "fallback_bbox": [77.45, 12.82, 77.80, 13.15],   # west, south, east, north (degrees)
    "analysis_crs": "EPSG:32643",                     # WGS 84 / UTM 43N, metres
    "web_crs": "EPSG:4326",
    "resolution_m": 30,
    "aoi_buffer_m": 5000,                              # margin for lakes and drains outside the city
    "buffer_deg": 0.06,                                # extra margin for rasters (slope, flow accumulation)

    # Elevation source: "FABDEM" (bare ground, recommended) or "COPERNICUS" (surface incl. buildings)
    "dem_source": "FABDEM",

    # September 2022 flood: radar images before and after
    "flood_event": {
        "pre_start": "2022-08-01", "pre_end": "2022-08-31",
        "post_start": "2022-09-04", "post_end": "2022-09-15"
    },

    # Hot season for the heat model (March to May)
    "heat_season": {"years": [2022, 2023, 2024], "start_month": 3, "end_month": 5},

    # Flood model weights (Phase 2). Must add up to 1.
    "flood_weights": {
        "elevation": 0.25, "slope": 0.10, "dist_lake": 0.20,
        "dist_drain": 0.20, "twi": 0.15, "builtup": 0.10
    },

    # >>> PASTE YOUR EARTH ENGINE CLOUD PROJECT ID HERE <<<
    "ee_project": "YOUR-EE-PROJECT-ID",

    "paths": {
        "project_dir": PROJECT_DIR,
        "raw": RAW,
        "processed": OUT,
        "gpkg": f"{OUT}/bengaluru.gpkg"
    }
}

assert abs(sum(CONFIG["flood_weights"].values()) - 1) < 1e-9, "Flood weights must add up to 1"
assert CONFIG["dem_source"] in ("FABDEM", "COPERNICUS")

with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('Saved config.json')

## 4. Connect to Google Earth Engine
Uses the project ID from the config cell.

The first time, a Google sign-in window opens. **Choose the Google account that owns your Earth Engine Cloud project**, and allow access. If you pick a different Google account, Earth Engine will refuse with a permission error. To switch accounts, set `FORCE_LOGIN = True` and rerun this cell.

In [ ]:
import ee, geemap

EE_PROJECT = CONFIG['ee_project']
if EE_PROJECT == 'YOUR-EE-PROJECT-ID':
    raise ValueError('Paste your Earth Engine project ID into CONFIG["ee_project"] first (section 3).')
FORCE_LOGIN = False                 # True = sign in again (e.g. you chose the wrong account)

def connect_ee():
    if FORCE_LOGIN:
        ee.Authenticate(force=True)
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        # Not logged in yet in this Colab session
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)

connect_ee()

# Test call: ask Earth Engine for something small
try:
    n = ee.ImageCollection('COPERNICUS/DEM/GLO30').size().getInfo()
    print(f'Earth Engine ready. Project: {EE_PROJECT}  (test: {n} DEM tiles in the catalog)')
except ee.EEException as err:
    msg = str(err)
    print('Earth Engine connected, but the test call failed:\n ', msg)
    if 'permission' in msg.lower() or 'not registered' in msg.lower():
        print('\nFix: sign in with the Google account that owns the project (set FORCE_LOGIN = True and rerun), '
              'and check that the project is registered for Earth Engine at '
              'https://code.earthengine.google.com/register')
    raise

## 5. City boundary and analysis area
The boundary comes from OpenStreetMap. If the lookup fails, or returns something that is not a polygon, the notebook falls back to a rectangle around the city so you can keep going.

The analysis area (`aoi`) is the boundary plus `aoi_buffer_m`. Lakes and drains are downloaded for the whole AOI.

In [ ]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import box

try:
    boundary = ox.geocode_to_gdf(CONFIG['osm_query'])[['geometry']]
    if not boundary.geom_type.isin(['Polygon', 'MultiPolygon']).all():
        raise ValueError(f'OSM returned {boundary.geom_type.iloc[0]}, not a polygon')
    source = 'OpenStreetMap'
except Exception as e:
    print('OSM lookup failed, using the fallback bounding box. Reason:', e)
    boundary = gpd.GeoDataFrame(geometry=[box(*CONFIG['fallback_bbox'])], crs='EPSG:4326')
    source = 'fallback bbox'

boundary = boundary.to_crs(CONFIG['web_crs'])
boundary['name'] = CONFIG['city']

area_km2 = boundary.to_crs(CONFIG['analysis_crs']).area.iloc[0] / 1e6
print(f'Boundary source: {source}')
print(f'Area: {area_km2:,.0f} km²')
if not 500 < area_km2 < 1000:
    print('WARNING: BBMP is roughly 700 to 750 km². Check the boundary on the map below.')

# Analysis area: city + buffer
aoi = gpd.GeoDataFrame(
    {'name': [f"{CONFIG['city']} + {CONFIG['aoi_buffer_m']/1000:g} km"]},
    geometry=boundary.to_crs(CONFIG['analysis_crs']).buffer(CONFIG['aoi_buffer_m']).values,
    crs=CONFIG['analysis_crs']
).to_crs(CONFIG['web_crs'])

ax = aoi.plot(edgecolor='grey', facecolor='none', linestyle='--', figsize=(6, 6))
boundary.plot(ax=ax, edgecolor='black', facecolor='none')
ax.set_title('City boundary (solid) and analysis area (dashed)');

## 6. Ward boundaries (optional)
Only needed later for the ward-level view. If the download fails, skip it; nothing else depends on it.

Note: the DataMeet file is the older 198-ward BBMP layout. Ward boundaries have since been redrawn (and in 2025 BBMP was replaced by the Greater Bengaluru Authority), so state which version you use in the report. GitHub raw links are case-sensitive, so the notebook tries a few spellings of the filename.

In [ ]:
WARDS_URLS = [
    'https://raw.githubusercontent.com/datameet/Municipal_Spatial_Data/master/Bangalore/BBMP.GeoJSON',
    'https://raw.githubusercontent.com/datameet/Municipal_Spatial_Data/master/Bangalore/BBMP.geojson',
]

wards = None
for url in WARDS_URLS:
    try:
        wards = gpd.read_file(url).to_crs(CONFIG['web_crs'])
        print(f'Loaded {len(wards)} wards from {url}')
        print('Columns:', list(wards.columns))
        wards.plot(edgecolor='grey', facecolor='none', figsize=(6, 6))
        break
    except Exception as e:
        print('Could not read', url, '->', type(e).__name__)

if wards is None:
    print('Ward download skipped. Find the file at https://github.com/datameet/Municipal_Spatial_Data and set WARDS_URLS.')

## 7. Lakes and drains from OpenStreetMap
Bengaluru floods mainly start at overflowing lakes and the stormwater drains that connect them, so these two layers drive the flood model.

`natural=wetland` is included because many of Bengaluru's lakes (or their marshy edges) are mapped that way. OSM's stormwater-drain network is incomplete; if you find an official network with a suitable licence, add it here later.

In [ ]:
aoi_poly = aoi.geometry.iloc[0]
city_poly = boundary.geometry.iloc[0]

def osm_layer(tags, geom_types, keep=('name',)):
    """Download OSM features inside the AOI, keep one geometry family and a few columns."""
    gdf = ox.features_from_polygon(aoi_poly, tags).reset_index()
    gdf = gdf[gdf.geom_type.isin(geom_types)].copy()
    cols = [c for c in keep if c in gdf.columns]
    gdf = gdf[cols + ['geometry']]
    gdf = gpd.clip(gdf, aoi)                       # trim to the analysis area
    gdf = gdf[gdf.geom_type.isin(geom_types)]      # clipping can create stray points
    gdf['in_city'] = gdf.intersects(city_poly)     # True if the feature touches the city
    return gdf.reset_index(drop=True)

# Lakes, tanks, reservoirs, wetlands
lakes = osm_layer(
    {'natural': ['water', 'wetland'], 'water': True, 'landuse': 'reservoir'},
    ['Polygon', 'MultiPolygon'],
    keep=('name', 'natural', 'water')
)
lakes['area_m2'] = lakes.to_crs(CONFIG['analysis_crs']).area.round(0)

# Drains, streams, canals, rivers
drains = osm_layer(
    {'waterway': ['river', 'stream', 'canal', 'drain', 'ditch']},
    ['LineString', 'MultiLineString'],
    keep=('name', 'waterway')
)
drains['length_m'] = drains.to_crs(CONFIG['analysis_crs']).length.round(0)

print(f'Lakes and water bodies: {len(lakes):,} '
      f'({lakes.in_city.sum():,} touch the city, total {lakes.area_m2.sum()/1e6:.1f} km²)')
print(f'Drains and waterways:   {len(drains):,} '
      f'({drains.in_city.sum():,} touch the city, total {drains.length_m.sum()/1e3:.0f} km)')
print(drains['waterway'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
aoi.plot(ax=ax, facecolor='none', edgecolor='grey', linestyle='--')
boundary.plot(ax=ax, facecolor='none', edgecolor='black')
drains.plot(ax=ax, color='steelblue', linewidth=0.6)
lakes.plot(ax=ax, color='dodgerblue')
ax.set_title('Bengaluru: lakes and drains (OpenStreetMap)')
ax.set_axis_off();

## 8. Save vector layers to the GeoPackage
The GeoPackage is first written to local Colab storage (fast, reliable), then copied to Drive. GeoServer will read this file later.

In [ ]:
LOCAL_GPKG = f'{WORK}/bengaluru.gpkg'
if os.path.exists(LOCAL_GPKG):
    os.remove(LOCAL_GPKG)

boundary.to_file(LOCAL_GPKG, layer='bbmp_boundary', driver='GPKG')
aoi.to_file(LOCAL_GPKG, layer='aoi', driver='GPKG')
lakes.to_file(LOCAL_GPKG, layer='lakes', driver='GPKG')
drains.to_file(LOCAL_GPKG, layer='drains', driver='GPKG')
if wards is not None:
    wards.to_file(LOCAL_GPKG, layer='wards', driver='GPKG')

shutil.copy(LOCAL_GPKG, CONFIG['paths']['gpkg'])
print('Layers saved:', gpd.list_layers(CONFIG['paths']['gpkg'])['name'].tolist())

## 9. Elevation and built-up rasters from Earth Engine
Both are exported on the same 30 m grid in UTM 43N, covering the AOI plus `buffer_deg`, so slope and flow calculations near the edge have the uphill land they need.

- **FABDEM** (default): Copernicus GLO-30 with buildings and trees removed, so it shows the ground, not rooftops. That matters for an urban flood model. Licence: CC BY-NC-SA 4.0 (fine for non-commercial work; credit it in the README). If FABDEM is unavailable, the cell falls back to Copernicus GLO-30.
- **Bilinear resampling** avoids the stepped artefacts that nearest-neighbour resampling leaves in the slope layer.
- **ESA WorldCover 2021**: the share of each 30 m cell that is built-up (0 to 1), used as the concrete/impervious factor.

In [ ]:
# Export area: AOI bounding box plus a margin
w, s, e, n = aoi.total_bounds
m = CONFIG['buffer_deg']
export_region = ee.Geometry.Rectangle([w - m, s - m, e + m, n + m])

def dem_from(collection_id):
    col = ee.ImageCollection(collection_id).filterBounds(export_region)
    return col, (col
                 .map(lambda img: img.select([0], ['elevation']).resample('bilinear'))
                 .mosaic())

dem_used = CONFIG['dem_source']
if dem_used == 'FABDEM':
    try:
        col, dem = dem_from('projects/sat-io/open-datasets/FABDEM')
        if col.size().getInfo() == 0:
            raise ValueError('no FABDEM tiles over the export region')
    except Exception as err:
        print('FABDEM not available, falling back to Copernicus GLO-30. Reason:', err)
        dem_used = 'COPERNICUS'
if dem_used == 'COPERNICUS':
    col, dem = dem_from('COPERNICUS/DEM/GLO30')
print('Elevation source:', dem_used)

# Built-up share: WorldCover class 50 = built-up, averaged from 10 m to 30 m
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
builtup = (worldcover.eq(50)
           .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
           .reproject(crs=CONFIG['analysis_crs'], scale=CONFIG['resolution_m'])
           .rename('builtup'))

def export_tif(image, name):
    """Download an EE image as GeoTIFF on the project grid, then copy it to Drive."""
    local = f'{WORK}/{name}.tif'
    geemap.ee_export_image(
        image.toFloat(),
        filename=local,
        scale=CONFIG['resolution_m'],
        crs=CONFIG['analysis_crs'],
        region=export_region,
        file_per_band=False,
    )
    shutil.copy(local, f'{RAW}/{name}.tif')
    print('Saved', f'{RAW}/{name}.tif')

export_tif(dem, 'dem_30m')
export_tif(builtup, 'builtup_30m')

# Record which DEM was actually used
CONFIG['dem_source_used'] = dem_used
with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

## 10. Check the rasters
Bengaluru sits on a plateau, so elevation should be roughly 780 to 960 m. Lakes should show up in the low areas.

Notebook 02 stacks these rasters cell by cell, so they must share exactly the same grid. The check below stops if they don't.

In [ ]:
import rasterio
import numpy as np

for name in ('dem_30m', 'builtup_30m'):
    with rasterio.open(f'{RAW}/{name}.tif') as src:
        arr = src.read(1, masked=True)
        print(f'{name}: {src.width} x {src.height} px, CRS {src.crs}, '
              f'res {src.res[0]:.0f} m, min {arr.min():.2f}, max {arr.max():.2f}')

with rasterio.open(f'{RAW}/dem_30m.tif') as a, rasterio.open(f'{RAW}/builtup_30m.tif') as b:
    same_grid = (a.shape == b.shape and a.transform.almost_equals(b.transform) and a.crs == b.crs)
assert same_grid, 'The two rasters are on different grids: rerun cell 9 or realign in Notebook 02.'
print('OK: both rasters share the same 30 m grid')

with rasterio.open(f'{RAW}/dem_30m.tif') as src:
    dem_arr = src.read(1, masked=True)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(dem_arr, cmap='terrain', extent=extent)
lakes.to_crs(CONFIG['analysis_crs']).plot(ax=ax, color='blue')
boundary.to_crs(CONFIG['analysis_crs']).plot(ax=ax, facecolor='none', edgecolor='black')
plt.colorbar(im, ax=ax, label='Elevation (m)', shrink=0.7)
ax.set_title(f'Elevation ({dem_used}) with lakes: water should sit in the low ground')
ax.set_axis_off();

## 11. Which Sentinel-1 radar images exist for the 2022 flood?
Only Sentinel-1A was working in 2022 (Sentinel-1B failed in December 2021), so Bengaluru was imaged roughly every 12 days per orbit. The water from the 4–5 September flood drained within days, so the post-flood image may miss much of it.

This cell lists every scene from August to September 2022. For the change detection in Notebook 04, the before and after images **must come from the same relative orbit**. The cell picks the best pair per orbit and saves the full list to `s1_scenes_2022.csv`.

In [ ]:
import pandas as pd

ev = CONFIG['flood_event']
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(export_region)
      .filterDate('2022-08-01', '2022-10-01')
      .filter(ee.Filter.eq('instrumentMode', 'IW')))

props = ['system:index', 'system:time_start', 'relativeOrbitNumber_start', 'orbitProperties_pass']
rows = s1.reduceColumns(ee.Reducer.toList(len(props)), props).get('list').getInfo()

scenes = pd.DataFrame(rows, columns=['id', 'time', 'orbit', 'pass'])
scenes['date'] = pd.to_datetime(scenes['time'], unit='ms').dt.date
scenes = scenes.drop(columns='time').sort_values(['date', 'orbit']).reset_index(drop=True)
print(f'{len(scenes)} Sentinel-1 IW scenes, Aug–Sep 2022:')
print(scenes[['date', 'orbit', 'pass', 'id']].to_string(index=False))

scenes.to_csv(f'{RAW}/s1_scenes_2022.csv', index=False)

# Best pre/post pair for each orbit
print('\nCandidate pairs (same orbit):')
pairs = []
for orbit, g in scenes.groupby('orbit'):
    gd = pd.to_datetime(g['date'])
    pre = g[gd.between(ev['pre_start'], ev['pre_end'])]
    post = g[gd.between(ev['post_start'], ev['post_end'])]
    if len(pre) and len(post):
        p, q = pre.iloc[-1], post.iloc[0]        # latest before, earliest after
        pairs.append({'orbit': int(orbit), 'pass': q['pass'],
                      'pre_date': str(p['date']), 'post_date': str(q['date']),
                      'pre_id': p['id'], 'post_id': q['id']})
        print(f"  orbit {orbit} ({q['pass']}): before {p['date']}  ->  after {q['date']}")

if not pairs:
    print('  None in the current window. Widen post_end in cell 3, and rely more on the news-report flood points.')
else:
    CONFIG['s1_pairs'] = pairs
    with open(f'{PROJECT_DIR}/config.json', 'w') as f:
        json.dump(CONFIG, f, indent=2)
    print('Saved the pairs to config.json for Notebook 04.')

## 12. Interactive check (optional)
A zoomable map to check the data by eye; it saves nothing. Zoom into Bellandur and Varthur in the southeast: these were among the worst-hit areas in September 2022.

- The background is Esri's light grey map. OpenStreetMap and CARTO block their free tiles inside Colab. For satellite imagery use `xyz.Esri.WorldImagery`.
- Elevation is shown over relief shading and clipped to the city; lakes and drains in the 5 km buffer are still shown.
- Only the display is clipped. The saved DEM keeps its margin, because slope and flow calculations at the city edge need the land around it.

In [ ]:
import xyzservices.providers as xyz

city_ee = ee.Geometry(boundary.geometry.iloc[0].__geo_interface__)

# Give the mosaic a real 30 m projection, so the shading is computed correctly
dem_p = dem.setDefaultProjection(crs=CONFIG['analysis_crs'], scale=CONFIG['resolution_m'])
hillshade = ee.Terrain.hillshade(dem_p)

elev_vis = {'min': 800, 'max': 950,
            'palette': ['1a9850', '91cf60', 'd9ef8b', 'fee08b', 'fc8d59', 'd73027']}

Map = geemap.Map(center=[12.97, 77.60], zoom=11, basemap=xyz.Esri.WorldGrayCanvas)
Map.add_layer(hillshade.clip(city_ee), {'min': 120, 'max': 220}, 'Hillshade', True, 1.0)
Map.add_layer(dem_p.clip(city_ee), elev_vis, f'Elevation ({dem_used})', True, 0.55)
Map.add_layer(worldcover.eq(50).selfMask().clip(city_ee), {'palette': ['555555']}, 'Built-up', False, 0.5)
Map.add_gdf(drains, layer_name='Drains', style={'color': '#2c7fb8', 'weight': 1})
Map.add_gdf(lakes, layer_name='Lakes',
            style={'color': '#08519c', 'weight': 0.5, 'fillColor': '#3182bd', 'fillOpacity': 0.8})
Map.add_gdf(aoi, layer_name='Analysis area (5 km buffer)',
            style={'color': '#888888', 'weight': 1, 'dashArray': '4', 'fillOpacity': 0})
Map.add_gdf(boundary, layer_name='City boundary',
            style={'color': '#000000', 'weight': 2, 'fillOpacity': 0})
Map.add_colorbar(elev_vis, label='Elevation (m)', layer_name=f'Elevation ({dem_used})')
Map

## 13. Summary

In [ ]:
print('Files in Drive:')
for folder in (RAW, OUT):
    for f in sorted(os.listdir(folder)):
        size = os.path.getsize(f'{folder}/{f}') / 1e6
        print(f'  {folder.split("bengaluru-climate-risk/")[1]}/{f:<24} {size:6.1f} MB')

print(f"\nElevation source used: {CONFIG.get('dem_source_used')}")
print(f"Sentinel-1 pairs found: {len(CONFIG.get('s1_pairs', []))}")
print('\nPhase 1 complete. Next: Notebook 02, the flood susceptibility model.')